# Routing benchmark v5 — fixing the MemoryError (for real this time)

## What v4 fixed vs. what actually crashed

v4 added a connected-component guard before `shortest_path_length`, to
avoid an expensive failed search on disconnected nodes. That fix is
still correct and still kept below — but it protects **section 8**
(the routing step). The traceback you hit happens earlier, in
**section 7**, inside `load_bezirk_graph`, on the line that does
`pickle.load(f)`. It never even reaches the routing step.

## What actually caused this crash

`load_bezirk_graph` caches every graph it loads into `graph_cache` and
never removes anything. The section-7 loop iterates over **all 12
Bezirke** (`groupby('bezirk_name')`), so each one gets loaded and kept
in memory permanently. OSMnx graphs carry a `geometry` (Shapely
LineString) on every edge, which is memory-heavy to unpickle. By the
time the loop reaches a dense Bezirk (Mitte, 5th alphabetically), the
combined memory of every previously-cached Bezirk graph plus the one
currently being unpickled exceeds available RAM — and the crash
surfaces mid-`pickle.load`, which is exactly what the traceback shows.

## The fix

Bound the cache: keep only the last `MAX_CACHED_GRAPHS` Bezirk graphs
in memory, evicting the oldest (and running `gc.collect()`) whenever
a new one is loaded. A `MemoryError` during load also now triggers a
full cache clear + one retry, as a last-resort safety net. This keeps
peak memory roughly constant regardless of how many Bezirke you
process, instead of growing with every new one.

## 1. Imports

If this is the first time using `igraph` in this environment, install
it once (free, open source): `pip install python-igraph` (the PyPI
package is named `python-igraph`, but you `import igraph`).

In [ ]:
import osmnx as ox
import networkx as nx
import igraph as ig
import numpy as np
import pandas as pd
import psycopg2
import pickle
import gc
import time
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv()

BEZIRKE_DIR = Path("C:\\Users\\PC\\Documents\\Projekt_Geodaten_haltung_Vernetzung\\data\\routing\\bezirke")
print("Available Bezirk networks (pickle):")
for f in sorted(BEZIRKE_DIR.glob("*.pkl")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f" - {f.name:35s} {size_mb:8.1f} MB on disk")


## 2. Connect to the database

In [3]:
conn = psycopg2.connect(
    host=os.getenv('DB_HOST', 'localhost'),
    database=os.getenv('DB_NAME', 'simra_inspired'),
    user=os.getenv('DB_USER', 'postgres'),
    password=os.getenv('DB_PASSWORD', '')
)
print("Connected.")


Connected.


## 3. Graph cache + connected-component map + igraph mirror (bounded memory, fast routing)

Three caching layers work together:

- `graph_cache` is bounded to `MAX_CACHED_GRAPHS` entries (LRU
  eviction) -- fixes the memory crash from before.
- `component_cache[id(G)]` maps every node to an island ID, so we
  never even attempt a search between nodes that can't be connected.
- `igraph_cache[id(G)]` holds a **mirror of the same graph rebuilt in
  igraph** (a C-backed graph library). This is the speed fix: igraph's
  Dijkstra is roughly one to two orders of magnitude faster than
  NetworkX's pure-Python implementation, and we call it once per ride
  with *all* the ride's node pairs batched together rather than once
  per segment -- so the routing step goes from thousands of small
  Python-level calls to a handful of vectorized C-level calls.

All three are evicted together whenever a Bezirk graph is dropped
from `graph_cache`, so memory stays bounded regardless of how many
Bezirke get processed.

In [ ]:
MAX_CACHED_GRAPHS = 3  # tune based on available RAM and file sizes above

graph_cache = {}          # name -> G, ordered by recency (dict preserves insertion order)
pair_cache = {}           # tuple(names) -> composed G
component_cache = {}      # id(G) -> {node: island_id}
igraph_cache = {}         # id(G) -> (igraph.Graph, {networkx_node: igraph_vertex_index})


def _drop_graph(name):
    """Remove a cached graph and anything derived from it, then collect."""
    G = graph_cache.pop(name, None)
    if G is not None:
        component_cache.pop(id(G), None)
        igraph_cache.pop(id(G), None)
    stale_pairs = [k for k in pair_cache if name in k]
    for k in stale_pairs:
        composed = pair_cache.pop(k)
        component_cache.pop(id(composed), None)
        igraph_cache.pop(id(composed), None)
    del G
    gc.collect()


def _evict_if_needed():
    while len(graph_cache) > MAX_CACHED_GRAPHS:
        oldest_name = next(iter(graph_cache))  # least-recently loaded
        print(f"    Evicting {oldest_name} from cache to free memory")
        _drop_graph(oldest_name)


def load_bezirk_graph(name):
    if name in graph_cache:
        G = graph_cache.pop(name)  # re-insert at the end -> most-recently-used
        graph_cache[name] = G
        return G

    path = BEZIRKE_DIR / f"{name}.pkl"
    t0 = time.time()
    try:
        with open(path, "rb") as f:
            G = pickle.load(f)
    except MemoryError:
        print(f"    MemoryError loading {name} -- clearing entire cache and retrying")
        graph_cache.clear()
        pair_cache.clear()
        component_cache.clear()
        igraph_cache.clear()
        gc.collect()
        with open(path, "rb") as f:
            G = pickle.load(f)

    graph_cache[name] = G
    print(f"    Loaded {name} in {time.time() - t0:.2f}s")
    _evict_if_needed()
    return G


def get_graph_for_bezirke(names):
    names = tuple(sorted(names))
    if len(names) == 1:
        return load_bezirk_graph(names[0])
    if names not in pair_cache:
        graphs = [load_bezirk_graph(n) for n in names]
        pair_cache[names] = nx.compose_all(graphs)
    return pair_cache[names]


def get_component_map(G):
    """
    Returns {node: island_id}, computed once per graph and cached.
    weakly_connected_components treats the graph as undirected for
    the purpose of finding "islands" -- appropriate here since we only
    care whether a path exists at all, regardless of direction.
    """
    key = id(G)
    if key not in component_cache:
        comp_map = {}
        for island_id, component in enumerate(nx.weakly_connected_components(G)):
            for node in component:
                comp_map[node] = island_id
        component_cache[key] = comp_map
        print(f"    Computed {len(set(comp_map.values()))} connected island(s) "
              f"for a graph with {len(G.nodes):,} nodes")
    return component_cache[key]


def get_igraph_for(G):
    """
    Rebuilds G as an igraph.Graph (C-backed) the first time it's needed,
    and caches it alongside the NetworkX version. Returns
    (igraph_graph, node_to_idx) where node_to_idx maps the original
    OSMnx/NetworkX node id to its integer index in the igraph graph.
    """
    key = id(G)
    if key not in igraph_cache:
        t0 = time.time()
        node_list = list(G.nodes())
        node_to_idx = {n: i for i, n in enumerate(node_list)}
        edges = []
        weights = []
        for u, v, data in G.edges(data=True):
            edges.append((node_to_idx[u], node_to_idx[v]))
            weights.append(float(data.get("length", 1.0)))
        Gi = ig.Graph(n=len(node_list), edges=edges, directed=True)
        Gi.es["length"] = weights
        igraph_cache[key] = (Gi, node_to_idx)
        print(f"    Built igraph mirror ({len(node_list):,} nodes, "
              f"{len(edges):,} edges) in {time.time() - t0:.2f}s")
    return igraph_cache[key]


## 4. Pick a small sample of rides

In [ ]:
# NOTE: the previous version of this query was
# "SELECT DISTINCT ride_id, random() AS r FROM gps_traces ORDER BY r LIMIT 30".
# Including random() in the SELECT list breaks DISTINCT (every row gets a
# different random value, so almost nothing gets deduplicated) -- Postgres
# ends up sorting the ENTIRE gps_traces table (millions of GPS points) just
# to pick 30 rows, which is what caused the OutOfMemory on the sort.
# Fix: deduplicate ride_id first (cheap), THEN randomize only over the much
# smaller set of distinct ride ids.
sample_rides = pd.read_sql('''
    SELECT ride_id
    FROM (SELECT DISTINCT ride_id FROM gps_traces) t
    ORDER BY random()
    LIMIT 30
''', conn)['ride_id'].tolist()

print(f"Sample size: {len(sample_rides)} rides")


## 5. Load GPS points and downsample

In [6]:
ride_ids_tuple = tuple(sample_rides)

points_df = pd.read_sql('''
    SELECT ride_id, lat, lon, "timeStamp"
    FROM gps_traces
    WHERE ride_id IN %(ride_ids)s
    ORDER BY ride_id, "timeStamp"
''', conn, params={"ride_ids": ride_ids_tuple})

TARGET_INTERVAL_SEC = 30

downsampled_rows = []
for ride_id, group in points_df.groupby("ride_id"):
    group = group.sort_values("timeStamp")
    kept = [group.iloc[0]]
    last_ts = group.iloc[0]["timeStamp"]
    for _, row in group.iloc[1:].iterrows():
        if (row["timeStamp"] - last_ts) / 1000.0 >= TARGET_INTERVAL_SEC:
            kept.append(row)
            last_ts = row["timeStamp"]
    downsampled_rows.extend(kept)

downsampled = pd.DataFrame(downsampled_rows).reset_index(drop=True)
print(f"Points after downsampling: {len(downsampled):,}")


C:\Users\PC\AppData\Local\Temp\ipykernel_21372\2872554659.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  points_df = pd.read_sql('''


Points after downsampling: 2,712


## 6. Find which Bezirk each point falls into

In [7]:
from psycopg2.extras import execute_values

with conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS temp_bench_points;")
    cur.execute('''
        CREATE TEMP TABLE temp_bench_points (
            row_id INTEGER,
            ride_id TEXT,
            lat DOUBLE PRECISION,
            lon DOUBLE PRECISION
        );
    ''')
    args = [
        (i, str(row['ride_id']), row['lat'], row['lon'])
        for i, row in downsampled.iterrows()
    ]
    execute_values(cur, '''
        INSERT INTO temp_bench_points (row_id, ride_id, lat, lon) VALUES %s
    ''', args)
    conn.commit()

bezirk_match = pd.read_sql('''
    SELECT p.row_id, b.name AS bezirk_name
    FROM temp_bench_points p
    JOIN bezirke b
        ON ST_Within(ST_SetSRID(ST_MakePoint(p.lon, p.lat), 4326), b.geom)
''', conn)

downsampled = downsampled.reset_index().rename(columns={"index": "row_id"})
downsampled = downsampled.merge(bezirk_match, on="row_id", how="left")

print(f"Points matched to a Bezirk: {downsampled['bezirk_name'].notna().sum()} / {len(downsampled)}")


C:\Users\PC\AppData\Local\Temp\ipykernel_21372\3711966427.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  bezirk_match = pd.read_sql('''


Points matched to a Bezirk: 2712 / 2712


## 7. Snap ALL points at once, grouped by Bezirk

In [8]:
t0 = time.time()

downsampled['node'] = None
for bezirk_name, group in downsampled.dropna(subset=['bezirk_name']).groupby('bezirk_name'):
    G = load_bezirk_graph(bezirk_name)
    nodes = ox.distance.nearest_nodes(G, X=group['lon'].values, Y=group['lat'].values)
    downsampled.loc[group.index, 'node'] = nodes

snap_time = time.time() - t0
print(f"\nSnapped all points (batched by Bezirk) in {snap_time:.2f}s")


    Loaded Charlottenburg-Wilmersdorf in 2.07s
    Loaded Friedrichshain-Kreuzberg in 1.93s
    Loaded Lichtenberg in 3.66s
    Loaded Marzahn-Hellersdorf in 4.72s
    Loaded Mitte in 4.98s
    Loaded Neukölln in 6.06s
    Loaded Pankow in 5.02s
    Loaded Reinickendorf in 1.65s
    Loaded Steglitz-Zehlendorf in 19.42s
    Loaded Tempelhof-Schöneberg in 1.39s


MemoryError: 

## 8. Compute routed distance per ride -- batched igraph Dijkstra

For each ride:

1. Skip segments whose endpoints are on different connected
   "islands" (v4 guard -- no path can exist).
2. Deduplicate the remaining source nodes and target nodes for the
   ride, then send **one batched call** `Gi.distances(source=unique_sources,
   target=unique_targets)` -- igraph requires the target list to have
   no duplicates, which is why we dedupe first. This computes one
   Dijkstra run per *unique* source (not per segment), which also cuts
   redundant work whenever a node repeats across segments.
3. Look up each segment's distance from the resulting matrix using
   the position of its (deduplicated) source and target.

In [ ]:
from tqdm.auto import tqdm


def routed_distance_for_ride(group):
    bezirke_used = set(group['bezirk_name'].dropna().unique())
    if not bezirke_used:
        return 0.0, 0, len(group) - 1

    G = get_graph_for_bezirke(bezirke_used)
    comp_map = get_component_map(G)
    Gi, node_to_idx = get_igraph_for(G)

    valid = group.dropna(subset=['node'])
    nodes = valid['node'].tolist()

    # Build the list of (u, v) segment pairs to route, skipping
    # anything that can't possibly have a path (connectivity guard).
    pair_u, pair_v = [], []
    n_failed = 0
    for i in range(len(nodes) - 1):
        u, v = nodes[i], nodes[i + 1]
        if u == v:
            continue
        if comp_map.get(u) != comp_map.get(v):
            n_failed += 1
            continue
        pair_u.append(u)
        pair_v.append(v)

    if not pair_u:
        return 0.0, 0, n_failed

    src_idx_all = [node_to_idx[u] for u in pair_u]
    tgt_idx_all = [node_to_idx[v] for v in pair_v]

    # igraph's distances() requires the target list to have no
    # duplicates (a node can legitimately be the target of more than
    # one segment in a ride) -- so dedupe both lists and look each
    # segment's distance up by position afterwards. This also avoids
    # rerunning Dijkstra from a source that repeats within the ride.
    unique_src = sorted(set(src_idx_all))
    unique_tgt = sorted(set(tgt_idx_all))
    src_pos = {v: i for i, v in enumerate(unique_src)}
    tgt_pos = {v: i for i, v in enumerate(unique_tgt)}

    dist_matrix = np.array(
        Gi.distances(source=unique_src, target=unique_tgt, weights="length")
    )

    total = 0.0
    n_segments = 0
    for s_idx, t_idx in zip(src_idx_all, tgt_idx_all):
        d = dist_matrix[src_pos[s_idx], tgt_pos[t_idx]]
        if np.isinf(d):
            n_failed += 1
        else:
            total += d
            n_segments += 1

    return total / 1000.0, n_segments, n_failed


t0 = time.time()
results = []
for ride_id, group in tqdm(downsampled.groupby("ride_id"),
                            total=downsampled["ride_id"].nunique(),
                            desc="Routing rides"):
    dist_km, n_seg, n_fail = routed_distance_for_ride(group)
    n_bezirke = group['bezirk_name'].nunique()
    results.append({
        "ride_id": ride_id,
        "routed_km": dist_km,
        "n_segments": n_seg,
        "n_failed": n_fail,
        "n_bezirke": n_bezirke,
    })
routing_time = time.time() - t0

results_df = pd.DataFrame(results)
print(f"Routed {len(results_df)} rides in {routing_time:.1f}s")
results_df


## 9. Sanity check vs straight-line distance

In [ ]:
straight_line = pd.read_sql('''
    SELECT ride_id, distance_km AS straight_line_km
    FROM ride_stats
    WHERE ride_id IN %(ride_ids)s
''', conn, params={"ride_ids": ride_ids_tuple})

comparison = results_df.merge(straight_line, on="ride_id")
comparison["ratio"] = comparison["routed_km"] / comparison["straight_line_km"]
comparison


## 10. Extrapolate total time for all 26,182 rides

In [ ]:
TOTAL_RIDES = 26182

avg_points_per_ride = len(downsampled) / len(sample_rides)
estimated_snap_time = (snap_time / len(downsampled)) * avg_points_per_ride * TOTAL_RIDES
estimated_routing_time = (routing_time / len(sample_rides)) * TOTAL_RIDES

estimated_total_sec = estimated_snap_time + estimated_routing_time
hours = estimated_total_sec / 3600

print(f"Estimated snapping time (all rides): {estimated_snap_time/3600:.2f}h")
print(f"Estimated routing time (all rides):  {estimated_routing_time/3600:.2f}h")
print(f"Estimated TOTAL time: {hours:.1f} hours")

if hours < 10:
    print("\nAn overnight run looks realistic.")
elif hours < 24:
    print("\nThis will take most of a day — consider increasing TARGET_INTERVAL_SEC.")
else:
    print("\nStill too slow — increase TARGET_INTERVAL_SEC and re-run this benchmark.")


## 11. Close connection

In [ ]:
conn.close()
print("Connection closed.")
